# Import Dependencies

In [155]:
# 1. Data Management (Always first)
%load_ext autoreload
%autoreload 2

# 2. Objects you are actually manipulating in cells
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 3. The Bridge to your script
from bender_functions import Bender

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Where do you want to save the data files?

In [156]:
# Add file paths, namse, and setup configuration
bender = Bender(config_module_name='jimenez_bender_config_A')
data_folder = r'c:\Users\jimen\Desktop\BenderData\2026-03-16_CodeTest'
base_name = '2026-03-16_CodeTest_'

#region  Create/show file path AND check configuration summary
outputfile = bender.increment_file_name(f"{data_folder}\\{base_name}.h5") # Create full file path
outputfig = outputfile.replace('.h5', '.png') # Mirror that name for the figure by simply swapping the extension
file_data = {
    "Type": ["HDF5 Data", "PNG Figure"],
    "Full System Path": [outputfile, outputfig]
}
df_files = pd.DataFrame(file_data) # Turn the Dictionary into a DataFrame Instance
pd.set_option('display.max_colwidth', None) # Tell Pandas not to cut off any text in the columns
display(df_files) # Display the Instance

# Print the summary of the Bender instance to check that everything is set up correctly
summary = bender.summary()
#endregion

Bender initialized using: jimenez_bender_config_A.py


,Type,Full System Path
0,HDF5 Data,c:\Users\jimen\Desktop\BenderData\2026-03-16_CodeTest\2026-03-16_CodeTest_001.h5
1,PNG Figure,c:\Users\jimen\Desktop\BenderData\2026-03-16_CodeTest\2026-03-16_CodeTest_001.png


              BENDER SYSTEM SUMMARY               
Config:      jimenez_bender_config_A.py
Device:      Dev1
Motor Port:  port0
Direction:   POSITIVE = LEFT
--------------------------------------------------
Cal File:    FT56491.cal
Sample Rate: 1000.0 Hz
Ramp:        0.25 s


## Biometrics: Input these before mounting

In [157]:

bender.update_metadata(
    fishcode = "foam", 
    segment = "anterior",
    fishmass = 100 ,   # Body mass in grams
    fishlen_TL = 186 ,     # Total length in mm
    fishlen_SL = 185 , # Standard length in mm

    # Mount-specific dimensions: input these after mounting but BEFORE running the experiment
    xsec_width = 50 ,       # mm Cross sectional width of fish between the clamps at the axis of rotation
    xsec_height = 83 ,       # mm Cross sectional height of fish between the clamps at the axis of rotation

    # Mounted biometrics
    dbend = 123 ,     # mm Distance from snout to the center of pressure ?
    dclamp = 50 ,     # mm Distance between the two clamps
    dvert = 190 ,         # mm Vertical distence from the transducer to the center of pressure
    dhoriz = 10          # mm Horizontal distence from the transducer to the center of pressure
)

  Stored: fishcode = foam
  Stored: segment = anterior
  Stored: fishmass = 100
  Stored: fishlen_TL = 186
  Stored: fishlen_SL = 185
  Stored: xsec_width = 50
  Stored: xsec_height = 83
  Stored: dbend = 123
  Stored: dclamp = 50
  Stored: dvert = 190
  Stored: dhoriz = 10


## Optional: if you want to drive bending motion based on muscle strain. 

In [158]:
# Use desired muscle strain to calculate the curvature amplitudes for bending motions.
desired_strain_pct = np.array([5])  # Desired strain in %
body_thickness = bender.xsec_width/1000   # Full thickness of the specimen in METERS. 
desired_curves = (2 * (desired_strain_pct / 100)) / body_thickness # Formula: Curvature (1/m) = (2 * Strain) / Thickness (m)

# What kind of experiment would you like to run?

If you want to drive curvature based on body thickness and red muscle strain, a simple combination can be performed to create inputs for all_curves.


In [159]:
bender.update_metadata(
    test_type = "dynamic", # Select your test type!

    # Bending motions
    all_freqs = [1], # Unlimited options for dynamic tests, but for frequency sweep these are start and end frequencies.
    all_curves = [1], # For dyanmic, ulimited. For sweeps, pick only one. Enter 'desired_curves' if you want to use the curvature calculated from the desired muscle strain.
    randomize = False,


    cycles_per_step = 5,    # How many times do you want to bend your specimen at each amplitude/frequency?
    n_end_cycles = 2,        # add cycles after last amplitude step
    stim_cycles_in_step = np.array([2,3]), # This array defines which cycles to activate
    # Bending motions for FREQUENCY SWEEP only (these will get ignore if you run a different test type)
    duration = 60,      # sec. How long to you want the whole test to last?
    amplitude_frequency_exponent = 0,   # should be between -1 and 0. Zero is constant amplitude, -1 is constant velocity, -0.5 is right in the middle.

    # Electrical stiulation
    is_stim = False, # True stimulates the muscle. False is for passive tests
    all_stimduties = [0],       # fractions of a cycle
    all_stimphases = [0],      # fractions of a cycle
    stim_pulse_rate = 75, # Hz shouldn't have to change this from 75
    
    # Change these to whatever the stimulator panel is set to.
    S1volts = 10, # Volts
    S2volts = 10, # Volts
    S1pulsedur = 2,          # ms
    S2pulsedur = 2           # ms
)

  Stored: test_type = dynamic
  Stored: all_freqs = [1]
  Stored: all_curves = [1]
  Stored: randomize = False
  Stored: cycles_per_step = 5
  Stored: n_end_cycles = 2
  Stored: stim_cycles_in_step = [2 3]
  Stored: duration = 60
  Stored: amplitude_frequency_exponent = 0
  Stored: is_stim = False
  Stored: all_stimduties = [0]
  Stored: all_stimphases = [0]
  Stored: stim_pulse_rate = 75
  Stored: S1volts = 10
  Stored: S2volts = 10
  Stored: S1pulsedur = 2
  Stored: S2pulsedur = 2


## Prepare experiment: takes all assigned values to create the experimental motions, stimuli, etc.

In [160]:
#region Run the calculation method to generate the sequence attributes. This applies only to some test_types (dynamic, static)
if bender.test_type in ['dynamic', 'static']:
    bender.organize_cycles(
        all_curves=bender.all_curves,
        all_freqs=bender.all_freqs,
        randomize=bender.randomize,
        cycles_per_step=bender.cycles_per_step,
        n_end_cycles=bender.n_end_cycles,
        dclamp=bender.dclamp,
        xsec_width=bender.xsec_width,
        stim_cycles_in_step=bender.stim_cycles_in_step,
        all_stimduties=bender.all_stimduties,
        all_stimphases=bender.all_stimphases,
        stim_pulse_rate=bender.stim_pulse_rate
    )

    # Generate the numbers for the plot
    angle, anglevel, tnorm, t = bender.make_dynamic_cycles(
        bender.period_by_cycle,
        bender.freq_by_cycle, 
        bender.amp_by_cycle
    )
    
    bender.record_motor_signal(t, angle, anglevel, tnorm)

elif bender.test_type in ['sweep']:
    bender.duration = duration # duration of each sweep in seconds
    bender.all_freqs = all_freqs # frequencies of sweep in Hz (should be a list of 2 values: [start, end])
    bender.all_curves = all_curves # amplitude of sweep in degrees
    bender.xsec_width = xsec_width
    bender.amplitude_frequency_exponent = amplitude_frequency_exponent # exponent for how amplitude changes
   
    # Generate the numbers for the plot
    angle, anglevel, tnorm, t = bender.make_frequency_sweep(
        bender.all_freqs, 
        bender.all_curves, 
        bender.amplitude_frequency_exponent, 
        bender.waitbefore
    )
#endregion


organize_cycles took 0.0 seconds


## CHECK: Is the experimental sequence what you wanted?

In [161]:
# Make a table
print("--- Generated Sequence Attributes ---")

display(pd.DataFrame({
    "freq (Hz)": bender.all_freqs,           # This is all_freqs_arr
    "curve (1/m)": bender.all_curves,       # This is all_curves_arr
    "amp (deg)": bender.all_degs,           # Expanded version of amps
    "strain (%)": bender.all_strains * 100,
    "strain rate (%/s)": bender.all_strainrates * 100,
    "duty (%)": bender.all_stimduties,      # Use the expanded array from the script
    "phase (%)": bender.all_stimphases      # Use the expanded array from the script
}))


# region MAKE PLOTS

# First plot: angle/stim plot with shaded stimulus periods
bender.make_stimuli()
fig = go.Figure()
fig.add_trace(go.Scatter(x=bender.t, y=bender.angle, name="Commanded Angle", line=dict(color='black')))

# Add Shading (ONLY if stimulation is active)
if bender.is_stim:
    # Handle Left Side Shading
    if hasattr(bender, 'Lonoff') and bender.Lonoff is not None:
        for onoff in bender.Lonoff:
            fig.add_vrect(x0=onoff[0], x1=onoff[1], fillcolor="blue", opacity=0.5, line_width=0, name="Left Stim")

    # Handle Right Side Shading
    if hasattr(bender, 'Ronoff') and bender.Ronoff is not None:
        for onoff in bender.Ronoff:
            fig.add_vrect(x0=onoff[0], x1=onoff[1], fillcolor="red", opacity=0.5, line_width=0, name="Right Stim")

fig.update_layout(title="Bending-Activation Preview", xaxis_title="Time (s)", yaxis_title="Angle (deg)")

display(fig)

# Second Plot: Muscle strain over time
kappa_over_time = np.deg2rad(bender.angle) / (bender.dclamp / 1000)  # Formula: curvature = angle_radians / length
strain_pct_over_time = ((kappa_over_time * (bender.xsec_width/1000)) / 2) * 100 # Formula: strain (%)= (curvature * thickness) / 2
fig_strain = go.Figure()

fig_strain.add_trace(go.Scatter(
    x=bender.t, 
    y=strain_pct_over_time, 
    name="Strain (%)", 
    line=dict(color='black')
))

# Add the same shading stim shading
if bender.is_stim:
    if hasattr(bender, 'Lonoff') and bender.Lonoff is not None:
        for onoff in bender.Lonoff:
            fig_strain.add_vrect(x0=onoff[0], x1=onoff[1], fillcolor="blue", opacity=0.3, line_width=0)
    if hasattr(bender, 'Ronoff') and bender.Ronoff is not None:
        for onoff in bender.Ronoff:
            fig_strain.add_vrect(x0=onoff[0], x1=onoff[1], fillcolor="red", opacity=0.3, line_width=0)

fig_strain.update_layout(
    title="Strain Over Time Preview", 
    xaxis_title="Time (s)", 
    yaxis_title="Strain (%)"
)
display(fig_strain)
# endregion

--- Generated Sequence Attributes ---


,freq (Hz),curve (1/m),amp (deg),strain (%),strain rate (%/s),duty (%),phase (%)
0,1,1,2.864789,2.5,15.707963,0,0


## Input dimensions to calculate Moment of Inertia for Clamps and Specimen

In [162]:
clamp_offset = 20.0 # mm Distance between the rotating clamps front margin and the axis of rotation
front_h, front_w = 160, 30
back_h, back_w = 20, 10 
spec_length = 100.0

# Set the physics for the bender based on the mounting dimensions and specimen dimensions. This will be used to calculate the moment of inertia and other parameters for the bending
bender.set_physics(clamp_offset, front_h, front_w, back_h, back_w, spec_length)


           PHYSICS CONFIGURATION REPORT           
Mode                      | lateral              
Total Rotating MOI        | 1540379.44   | g*mm²
Lever Arm (r)             | 30.00        | mm
Specimen Mass             | 383.50       | g
--------------------------------------------------
TOTAL SYSTEM MASS         | 807.98       | g



## CALCULATE: important measurements to be included in the data file (H5).

In [163]:
#Theoretically not necessary since most values are saved, but simplifies pipeline.  
#test_section_pos = dbend/(fishlen_TL + dclamp + CLAMP_D)   # Position of the bending segment at the axis of rotation (fraction of total length). Assuming the clamp depth (CLAMP_D below) is 20. 


# Make sure to save stim data!! Should look something like this:
# ADD STIMULI (code will ignore if is_stim is False)
# 1. Generate the stimulus signals
#S1stimcmd, S2stimcmd = bender.make_stimuli()

# 2. Extract the Lonoff and Ronoff lists the function just saved
#Lonoff = bender.Lonoff
#Ronoff = bender.Ronoff

# START BENDING!! 

This is the main code block that runs the experiment. It sets up the DAQ, sends the output, records the input, and writes it to the file.

In [164]:
bender.run_experiment(test_type="dynamic")
print(f"Total pulses generated: {np.sum(bender.dig > 0)}")

Data will be saved to: experiment_data_dynamic_000001.h5
📏 Sonometer (Left) Calibrated: 30.31 mm
📏 Sonometer (Right) Calibrated: 0.51 mm
Total pulses generated: 840000


In [165]:
#region Save the actual text of your config file for ultimate traceability
with h5py.File(outputfile, 'w') as f:

    with open(f"{bender.config_name}.py", 'r') as cfg_file:
        f.attrs['Config_File_Content'] = cfg_file.read()
        f.attrs['Config_Module_Used'] = bender.config_name

    # --- Group 1: General Experiment Information ---
    g_info = f.create_group('ExperimentInfo')
    g_info.attrs['EndTime'] = bender.endTime.strftime('%Y-%m-%d %H:%M:%S %Z')
    g_info.attrs['FishCode'] = fishcode
    g_info.attrs['Segment'] = segment

    # --- Group 2: Specimen Geometry and Setup ---
    g_specimen = f.create_group('Biometrics')
    g_specimen.attrs['FishLength_mm'] = fishlen
    g_specimen.attrs['FishMass_g'] = fishmass
    g_specimen.attrs['FishCrossSectionWidth_mm'] = xsec_width
    g_specimen.attrs['FishCrossSectionHeight_mm'] = xsec_height
    g_specimen.attrs['TestSectionPosition_perc'] = test_section_pos

    # --- Group 3: Mount Geometry ---
    g_mount = f.create_group('MountGeometry')
    g_mount.attrs['BendLocation_mm'] = dbend
    g_mount.attrs['ClampDistance_mm'] = dclamp
    g_mount.attrs['DistanceFromTransducerVert_mm'] = dvert
    g_mount.attrs['DistanceFromTransducerHoriz_mm'] = dhoriz

    # --- Group 4: Inertial/Mass Properties (MOI) ---
    g_moi = f.create_group('InertialProperties')
    g_moi.attrs['TotalSystemMass_g'] = Total_Mass_System
    g_moi.attrs['TotalSystemMOI_gmm2'] = I_total_system

    # --- Group 5: Physical Dimensions used for calculations ---
    g_dims = f.create_group('CalculationDimensionsMOI')
    g_dims.attrs['Clamp_Height_mm'] = CLAMP_H
    g_dims.attrs['Clamp_Width_mm'] = CLAMP_W
    g_dims.attrs['Clamp_Depth_mm'] = CLAMP_D
    g_dims.attrs['Clamp_Density_gmm3'] = RHO_CLAMP
    g_dims.attrs['Specimen_Height_mm'] = SPECIMEN_H
    g_dims.attrs['Specimen_Depth_mm'] = SPECIMEN_D
    g_dims.attrs['Specimen_FrontHeight_mm'] = FRONT_H
    g_dims.attrs['Specimen_FrontWidth_mm'] = FRONT_W
    g_dims.attrs['Specimen_BackHeight_mm'] = BACK_H
    g_dims.attrs['Specimen_BackWidth_mm'] = BACK_W
    g_dims.attrs['Specimen_Density_gmm3'] = RHO_OBJECT 

    # Start saving raw data, calibrated data, and output data
    gin = f.create_group('RawInput')
    gin.attrs['SampleFrequency'] = samplefreq
  
    # Store measurements
    gin.create_dataset('forcetransducer', data=aidata[:6,:])
    gin.create_dataset('Stimulation_monitor', data=aidata[6,:])

    gcal = f.create_group('Calibrated')
    for ft1, name1 in zip(forcetorque, forcetorque_names):
        gcal.create_dataset(name1, data=ft1)
    gcal.create_dataset('CalibrationMatrix', data=bender.calibration) # Save the calibration matrix used

    ds = gcal.create_dataset('Encoder', data=bender.angledata) # DOUBLE CHECK THIS
    ds.attrs['CountsPerRev'] = encoder_counts_per_rev

    # save the output data
    gout = f.create_group('Output')
    gout.attrs['SampleFrequency'] = outputfreq
    gout.create_dataset('DigitalOut', data=dig)
    gout.create_dataset('SyncInTrainDur', data=S1actcmd)
    gout.create_dataset('SyncInS2Del', data=S2actcmd)
    gout.attrs['S1side'] = S1side
    gout.attrs['S2side'] = S2side
    gout.attrs['S1volts'] = S1volts
    gout.attrs['S2volts'] = S2volts
    gout.attrs['S1pulsedur_ms'] = S1pulsedur    
    gout.attrs['S2pulsedur_ms'] = S2pulsedur    
    
    # Save stimulus parameters
    gout = f.create_group('NominalStimulus')
    gout.attrs['Type'] = 'Dynamic'

    gout.create_dataset('t', data=t)
    ds = gout.create_dataset('Position', data=angle)
    ds.attrs['Units'] = 'deg'
    ds = gout.create_dataset('Velocity', data=anglevel)
    ds.attrs['Units'] = 'deg/sec'
    gout.create_dataset('tnorm', data=tnorm)
    gout.create_dataset('Lonoff', data=Lonoff)
    gout.create_dataset('Ronoff', data=Ronoff)

    # Save bending parameters
    # Think about how to save all amps, curves, strains, velocities, etc.
    all_amps = []  # TODO: Replace with actual amplitude data if available
    gout.attrs['Curvatures'] = all_curves
    gout.attrs['Frequencies'] = all_freqs
    gout.attrs['CyclesPerStep'] = cycles_per_step
    gout.attrs['EndCycles'] = n_end_cycles
    gout.attrs['FrequencyByCycle'] = freq_by_cycle
    gout.attrs['AmplitudeByCycle'] = amp_by_cycle
    gout.attrs['IsStimByCycle'] = is_stim_cycle
    gout.attrs['CycleRandomOrder'] = order
    gout.attrs['MovementDuration'] = movedur

    # Add estimated red muscle strain and strain rate
    gout.attrs['Strains'] = allstrains
    gout.attrs['StrainRates'] = allstrainrates  

    gout.attrs['WaitPre'] = waitbefore
    gout.attrs['WaitPost'] = waitafter
    gout.attrs['PrePostStimDur'] = prepoststim_dur
    gout.attrs['ScaleFactor'] = scale
    gout.attrs['PositiveMotorDirection'] = positive_motor_direction

    gout.attrs['StimulationOn'] = is_stim
    gout.attrs['StimulationDuty'] = all_stimduties
    gout.attrs['StimulationPhase'] = all_stimphases
    gout.attrs['StimulationPulseRate'] = stim_pulse_rate

    #endregion


NameError: name 'h5py' is not defined

##  Zero the data for plotting results

In [ ]:
for ft1 in forcetorque:
    ft1 -= np.mean(ft1[t < 0])

NameError: name 'forcetorque' is not defined

## QUALITY CONTROL: Plot control signals AND raw torque over time

In [ ]:
print(bender.aidata)

[[-0.00929236 -0.00929236 -0.0088093  ... -0.01009747 -0.00977543
  -0.01041952]
 [-0.08497247 -0.08690472 -0.08658268 ... -0.0867437  -0.08642166
  -0.08046386]
 [ 0.11759262  0.11759262  0.11759262 ...  0.11952488  0.11727057
   0.1174316 ]
 ...
 [ 0.00149608  0.02049662  0.02999689 ...  0.00874205  0.02838667
   0.03015791]
 [ 0.01260656  0.01260656  0.01244554 ...  0.8149778   0.81594393
   0.81642699]
 [ 0.01357269  0.01373371  0.01389473 ...  0.01405576  0.01389473
   0.01421678]]


In [168]:
raw_v = bender.aidata[6,0] # First sample of the sono channel
cal_mm = bender.apply_calibration_sono(raw_v, bender.sono_cal_left)

print(f"Voltage at start: {raw_v:.3f} V")
print(f"Calculated distance: {cal_mm:.3f} mm")

Voltage at start: 2.316 V
Calculated distance: 24.393 mm


In [170]:
# Make sure to plot the correct bending axis. It depends on sensor mounting and orientation!
fig = make_subplots(rows = 4, cols = 1,
                   shared_xaxes=True)
#fig.add_trace(
 #   go.Scatter(x = tnorm, y = angle_measured, mode="lines", name="angle_enc"),
  #  row=1, col=1)
fig.add_trace(
    go.Scatter(x = bender.t, y = bender.angle, mode="lines", name="angle_cmd"),
    row=1, col=1)
fig.add_trace(
    go.Scatter(x = bender.t, y = bender.sono_left_mm, mode="lines", name="sono1_mm"),
    row=4, col=1)
fig.add_trace(
    go.Scatter(x = bender.t, y = bender.forcetorque[3,:], mode="lines", name="Tx"),
    row=2, col=1)
# fig.add_trace(
#     go.Scatter(x = tnorm, y = bender.forcetorque[1,:], mode="lines", name="Fy"),
#     row=4, col=1)
fig.add_trace(
    go.Scatter(x = bender.t, y = bender.forcetorque[5,:], mode="lines", name="Tz"),
    row=3, col=1)


fig.update_yaxes(title_text = "angle (deg)", row=1)
fig.update_yaxes(title_text = "Tx (Nm)", row=2)
fig.update_yaxes(title_text = "Tz (Nm)", row=3)
fig.update_yaxes(title_text = "sono1 (mm)", row=4)
fig.update_xaxes(title_text = "time (s)", row=3)
fig.update_layout(title_text = bender.filename)

# Save the plot in output folder
#fig.write_image(outputfig, format= 'png', width=1200, height=800)


## QUALITY CONTROL: Do the loops like nice?

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=angle, y=forcetorque[3,:]))
fig.update_yaxes(title_text="torque (Nm)")
fig.update_xaxes(title_text="angle (deg)")